# Análisis de ventas con pandas

Resolución de los 12 ejercicios de la hoja **Ejercicios** del archivo `practica_analisis_datos_empresariales.xlsx`.

**Contexto:** TecnoAndes vende productos de tecnología, oficina y software en 4 regiones. Los datos (ficticios) van de enero 2025 a octubre 2026 y los montos están en USD.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.options.display.float_format = "{:,.2f}".format

# El notebook busca el Excel en la carpeta actual o en la carpeta superior (repositorio)
nombre = "practica_analisis_datos_empresariales.xlsx"
ruta = next(p for p in (Path(nombre), Path("..") / nombre) if p.exists())
df = pd.read_excel(ruta, sheet_name="Ventas")
df.head()

## 1. Revisión y limpieza

In [ ]:
df.info()
print("Nulos por columna:\n", df.isna().sum())
print("Filas duplicadas:", df.duplicated().sum())

In [ ]:
# Nombres de columnas más cómodos para programar
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"\s*\(usd\)", "", regex=True)
              .str.replace(" ", "_"))

df = df.drop_duplicates()
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
for col in ["región", "vendedor", "producto", "categoría"]:
    df[col] = df[col].str.strip()
df = df.dropna(subset=["fecha", "unidades"])

# Las columnas calculadas llegan desde Excel; las recalculamos para no depender de ellas
df["ingreso"] = df["unidades"] * df["precio_unitario"] * (1 - df["descuento"])
df["costo_total"] = df["unidades"] * df["costo_unitario"]
df["margen"] = df["ingreso"] - df["costo_total"]
df["margen_pct"] = df["margen"] / df["ingreso"]

print(df.shape)
df.columns.tolist()

## Ejercicio 1. Ingreso total del período

In [ ]:
ingreso_total = df["ingreso"].sum()
print(f"Ingreso total: USD {ingreso_total:,.0f}")

## Ejercicio 2. Margen % total

Se calcula como margen total / ingreso total. No se promedia la columna `margen_pct`, porque eso daría el mismo peso a ventas grandes y pequeñas.

In [ ]:
margen_total = df["margen"].sum()
print(f"Margen total: USD {margen_total:,.0f}")
print(f"Margen %: {margen_total / ingreso_total:.1%}")

## Ejercicio 3. Región con mayor ingreso

In [ ]:
por_region = df.groupby("región")["ingreso"].sum().sort_values(ascending=False)
print(por_region)
print("\nMayor ingreso:", por_region.idxmax())

## Ejercicio 4. Ventas con margen negativo

In [ ]:
negativas = df[df["margen"] < 0]
print("Ventas con margen negativo:", len(negativas))
negativas[["fecha", "producto", "descuento", "ingreso", "margen"]]

## Ejercicio 5. Producto con más unidades vendidas

In [ ]:
por_producto = df.groupby("producto")["unidades"].sum().sort_values(ascending=False)
print(por_producto)
print("\nMás vendido:", por_producto.idxmax())

## Ejercicio 6. Vendedor con mayor margen (USD)

In [ ]:
por_vendedor = df.groupby("vendedor")["margen"].sum().sort_values(ascending=False)
print(por_vendedor)
print("\nMayor margen:", por_vendedor.idxmax())

## Ejercicio 7. Ingreso del tercer trimestre de 2026

In [ ]:
q3 = df[(df["fecha"] >= "2026-07-01") & (df["fecha"] <= "2026-09-30")]
print(f"Ingreso Q3 2026: USD {q3['ingreso'].sum():,.0f}")

## Ejercicio 8. Descuento promedio (solo ventas con descuento)

In [ ]:
con_desc = df[df["descuento"] > 0]
print(f"Descuento promedio: {con_desc['descuento'].mean():.1%}")
print("Ventas con descuento:", len(con_desc), "de", len(df))

## Ejercicio 9. Categoría con mayor margen %

In [ ]:
cat = df.groupby("categoría")[["margen", "ingreso"]].sum()
cat["margen_pct"] = cat["margen"] / cat["ingreso"]
print(cat.sort_values("margen_pct", ascending=False))
print("\nMayor margen %:", cat["margen_pct"].idxmax())

## Ejercicio 10. Crecimiento enero–octubre 2026 vs. 2025

In [ ]:
def ingreso_periodo(anio):
    m = (df["fecha"] >= f"{anio}-01-01") & (df["fecha"] <= f"{anio}-10-31")
    return df.loc[m, "ingreso"].sum()

i25, i26 = ingreso_periodo(2025), ingreso_periodo(2026)
print(f"Ene–oct 2025: USD {i25:,.0f}")
print(f"Ene–oct 2026: USD {i26:,.0f}")
print(f"Crecimiento: {i26 / i25 - 1:.1%}")

## Ejercicio 11. Ingresos por mes

In [ ]:
mensual = df.groupby(df["fecha"].dt.to_period("M"))["ingreso"].sum()
mensual.index = mensual.index.astype(str)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(mensual.index, mensual.values, marker="o")
ax.set_title("Ingreso mensual (USD)")
ax.set_ylabel("USD")
plt.xticks(rotation=60)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Tendencia observada:** *(escribe aquí en una frase lo que ves en el gráfico: crecimiento, caídas, estacionalidad)*

## Ejercicio 12. Recomendación para la gerencia

Apóyate en los resultados anteriores. Este gráfico y esta tabla muestran cómo el descuento afecta el margen.

In [ ]:
df["tramo_desc"] = pd.cut(df["descuento"], [-0.01, 0, 0.10, 0.20, 0.30],
                          labels=["0%", "1–10%", "11–20%", "21–30%"])
efecto = df.groupby("tramo_desc", observed=True)[["margen", "ingreso"]].sum()
efecto["margen_pct"] = efecto["margen"] / efecto["ingreso"]
print(efecto)

efecto["margen_pct"].plot(kind="bar", title="Margen % según tramo de descuento", rot=0)
plt.ylabel("Margen %")
plt.show()

**Recomendación:** *(escribe aquí 2–3 líneas: por ejemplo, si conviene limitar descuentos altos en ciertos productos o categorías)*

## Guardar resultados

In [ ]:
Path("../output").mkdir(exist_ok=True)
df.to_csv("../output/ventas_limpias.csv", index=False)
print("Guardado: output/ventas_limpias.csv")